# LangGraph and LangSmith - Agentic RAG Powered by LangChain

In the following notebook we'll complete the following tasks:

- 🤝 Breakout Room #1:
  1. Install required libraries
  2. Set Environment Variables
  3. Creating our Tool Belt
  4. Creating Our State
  5. Creating and Compiling A Graph!

- 🤝 Breakout Room #2:
  1. Evaluating the LangGraph Application with LangSmith
  2. Adding Helpfulness Check and "Loop" Limits
  3. LangGraph for the "Patterns" of GenAI

# 🤝 Breakout Room #1

## Part 1: LangGraph - Building Cyclic Applications with LangChain

LangGraph is a tool that leverages LangChain Expression Language to build coordinated multi-actor and stateful applications that includes cyclic behaviour.

### Why Cycles?

In essence, we can think of a cycle in our graph as a more robust and customizable loop. It allows us to keep our application agent-forward while still giving the powerful functionality of traditional loops.

Due to the inclusion of cycles over loops, we can also compose rather complex flows through our graph in a much more readable and natural fashion. Effectively allowing us to recreate application flowcharts in code in an almost 1-to-1 fashion.

### Why LangGraph?

Beyond the agent-forward approach - we can easily compose and combine traditional "DAG" (directed acyclic graph) chains with powerful cyclic behaviour due to the tight integration with LCEL. This means it's a natural extension to LangChain's core offerings!

## Task 1:  Dependencies


## Task 2: Environment Variables

We'll want to set both our OpenAI API key and our LangSmith environment variables.

In [2]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

In [3]:
os.environ["TAVILY_API_KEY"] = getpass.getpass("TAVILY_API_KEY")

In [4]:
from uuid import uuid4

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = f"AIE7 - LangGraph - {uuid4().hex[0:8]}"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangSmith API Key: ")

## Task 3: Creating our Tool Belt

As is usually the case, we'll want to equip our agent with a toolbelt to help answer questions and add external knowledge.

There's a tonne of tools in the [LangChain Community Repo](https://github.com/langchain-ai/langchain-community/tree/main/libs/community) but we'll stick to a couple just so we can observe the cyclic nature of LangGraph in action!

We'll leverage:

- [Tavily Search Results](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/tavily_search/tool.py)
- [Arxiv](https://github.com/langchain-ai/langchain-community/blob/main/libs/community/langchain_community/tools/arxiv/tool.py)

#### 🏗️ Activity #1:

Please add the tools to use into our toolbelt.

> NOTE: Each tool in our toolbelt should be a method.

##### ✅ Answer:

To complete this task, I'll add my own tool for fun. It will answer the ultimate questions of the universe. I'll label cells related to this activity with a "✅🏗️ Activity #1:" header, or a comment if I'm adding to existing code.

In [5]:
from langchain_core.tools import Tool

def answer_questions_of_the_universe(question: str) -> str:
    return "the answer is 42"

universe_tool = Tool(
    name="answer_questions_of_the_universe",
    func=answer_questions_of_the_universe,
    description="Always returns the answer to the ultimate question of life, the universe, and everything."
)

In [6]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools.arxiv.tool import ArxivQueryRun

tavily_tool = TavilySearchResults(max_results=5)

tool_belt = [
    tavily_tool,
    ArxivQueryRun(),
    universe_tool, # ✅🏗️ Activity #1
]

/var/folders/ns/n39s_yzn62zby9b115fkzw2h0000gn/T/ipykernel_16636/612645377.py:4: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-tavily package and should be used instead. To use it run `pip install -U :class:`~langchain-tavily` and import as `from :class:`~langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(max_results=5)


### Model

Now we can set-up our model! We'll leverage the familiar OpenAI model suite for this example - but it's not *necessary* to use with LangGraph. LangGraph supports all models - though you might not find success with smaller models - as such, they recommend you stick with:

- OpenAI's GPT-3.5 and GPT-4
- Anthropic's Claude
- Google's Gemini

> NOTE: Because we're leveraging the OpenAI function calling API - we'll need to use OpenAI *for this specific example* (or any other service that exposes an OpenAI-style function calling API.

In [7]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

Now that we have our model set-up, let's "put on the tool belt", which is to say: We'll bind our LangChain formatted tools to the model in an OpenAI function calling format.

In [8]:
model = model.bind_tools(tool_belt)

#### ❓ Question #1:

How does the model determine which tool to use?

##### ✅ Answer:

The model utilizes a description of the tools along with some input to determine which tool to use. The description includes things like what the tool does and what arguments it expects.

As a very basic example, if we supplied a model with an `add` tool with the description "Adds two numbers together. Expects arguments `a`: the first number, and `b`: the second number" and a user input like "What's four plus two?" The LLM would look at that tool and say, great! Let's use `add` with `4` as `a` and `2` as `b`.

Of course, this input is much more structured and often includes more context and tools, but the minimal example is a nice mental model.

This is done by leveraging of OpenAI's function calling API.

## Task 4: Putting the State in Stateful

Earlier we used this phrasing:

`coordinated multi-actor and stateful applications`

So what does that "stateful" mean?

To put it simply - we want to have some kind of object which we can pass around our application that holds information about what the current situation (state) is. Since our system will be constructed of many parts moving in a coordinated fashion - we want to be able to ensure we have some commonly understood idea of that state.

LangGraph leverages a `StatefulGraph` which uses an `AgentState` object to pass information between the various nodes of the graph.

There are more options than what we'll see below - but this `AgentState` object is one that is stored in a `TypedDict` with the key `messages` and the value is a `Sequence` of `BaseMessages` that will be appended to whenever the state changes.

Let's think about a simple example to help understand exactly what this means (we'll simplify a great deal to try and clearly communicate what state is doing):

1. We initialize our state object:
  - `{"messages" : []}`
2. Our user submits a query to our application.
  - New State: `HumanMessage(#1)`
  - `{"messages" : [HumanMessage(#1)}`
3. We pass our state object to an Agent node which is able to read the current state. It will use the last `HumanMessage` as input. It gets some kind of output which it will add to the state.
  - New State: `AgentMessage(#1, additional_kwargs {"function_call" : "WebSearchTool"})`
  - `{"messages" : [HumanMessage(#1), AgentMessage(#1, ...)]}`
4. We pass our state object to a "conditional node" (more on this later) which reads the last state to determine if we need to use a tool - which it can determine properly because of our provided object!

In [9]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages
import operator
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

## Task 5: It's Graphing Time!

Now that we have state, and we have tools, and we have an LLM - we can finally start making our graph!

Let's take a second to refresh ourselves about what a graph is in this context.

Graphs, also called networks in some circles, are a collection of connected objects.

The objects in question are typically called nodes, or vertices, and the connections are called edges.

Let's look at a simple graph.

![image](https://i.imgur.com/2NFLnIc.png)

Here, we're using the coloured circles to represent the nodes and the yellow lines to represent the edges. In this case, we're looking at a fully connected graph - where each node is connected by an edge to each other node.

If we were to think about nodes in the context of LangGraph - we would think of a function, or an LCEL runnable.

If we were to think about edges in the context of LangGraph - we might think of them as "paths to take" or "where to pass our state object next".

Let's create some nodes and expand on our diagram.

> NOTE: Due to the tight integration with LCEL - we can comfortably create our nodes in an async fashion!

In [10]:
from langgraph.prebuilt import ToolNode

def call_model(state):
  messages = state["messages"]
  response = model.invoke(messages)
  return {"messages" : [response]}

tool_node = ToolNode(tool_belt)

Now we have two total nodes. We have:

- `call_model` is a node that will...well...call the model
- `tool_node` is a node which can call a tool

Let's start adding nodes! We'll update our diagram along the way to keep track of what this looks like!


In [11]:
from langgraph.graph import StateGraph, END

uncompiled_graph = StateGraph(AgentState)

uncompiled_graph.add_node("agent", call_model)
uncompiled_graph.add_node("action", tool_node)

Let's look at what we have so far:

![image](https://i.imgur.com/md7inqG.png)

Next, we'll add our entrypoint. All our entrypoint does is indicate which node is called first.

In [12]:
uncompiled_graph.set_entry_point("agent")

![image](https://i.imgur.com/wNixpJe.png)

Now we want to build a "conditional edge" which will use the output state of a node to determine which path to follow.

We can help conceptualize this by thinking of our conditional edge as a conditional in a flowchart!

Notice how our function simply checks if there is a "function_call" kwarg present.

Then we create an edge where the origin node is our agent node and our destination node is *either* the action node or the END (finish the graph).

It's important to highlight that the dictionary passed in as the third parameter (the mapping) should be created with the possible outputs of our conditional function in mind. In this case `should_continue` outputs either `"end"` or `"continue"` which are subsequently mapped to the action node or the END node.

In [13]:
def should_continue(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  return END

uncompiled_graph.add_conditional_edges(
    "agent",
    should_continue
)

Let's visualize what this looks like.

![image](https://i.imgur.com/8ZNwKI5.png)

Finally, we can add our last edge which will connect our action node to our agent node. This is because we *always* want our action node (which is used to call our tools) to return its output to our agent!

In [14]:
uncompiled_graph.add_edge("action", "agent")

Let's look at the final visualization.

![image](https://i.imgur.com/NWO7usO.png)

All that's left to do now is to compile our workflow - and we're off!

In [15]:
simple_agent_graph = uncompiled_graph.compile()

#### ❓ Question #2:

Is there any specific limit to how many times we can cycle?

If not, how could we impose a limit to the number of cycles?

##### ✅ Answer:

There's no explicit limit set on the number of cycles.

For the sake of your wallet, and the likely declining performance after a certain number of cycles and an increasing amount of context, we should probably always place a limit on the number of cycles, even if arbitrary.

To do so, we can simply count the number of messages within our state, and return `END` if we exceed some limit. The number of messages won't be a direct mapping to the number of cycles, but the number of messages does _increase_ with each cycle, and therefore would work well for ensuring we don't end up in an infinite loop.

If wanted precisely to limit the number of cycles, we could think about a solution where we count the number of times our agent is invoked and _doesn't_ `END`. This marks the beginning of a cycle. Again, if we've reached that limit, we return `END`.

## Using Our Graph

Now that we've created and compiled our graph - we can call it *just as we'd call any other* `Runnable`!

Let's try out a few examples to see how it fairs:

In [16]:
# ✅🏗️ Activity #1

from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="What is the ultimate answer to the universe?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_lrbQJtMiLj7PSlrHc4LlQntf', 'function': {'arguments': '{"__arg1":"What is the ultimate answer to the universe?"}', 'name': 'answer_questions_of_the_universe'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 28, 'prompt_tokens': 203, 'total_tokens': 231, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-Bsp1PNbL8UjwPQqO69KoQODeCBkIg', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--36065da2-9d60-4486-b201-f6e5385756be-0', tool_calls=[{'name': 'answer_questions_of_the_universe', 'args': {'__arg1': 'What is the ultimate answer to the universe?'}, 'id': 'call_

In [17]:
from langchain_core.messages import HumanMessage

inputs = {"messages" : [HumanMessage(content="Who is the current captain of the Winnipeg Jets?")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_B6oBeUQ0crL0oZWbFb6MaZZM', 'function': {'arguments': '{"query":"current captain of the Winnipeg Jets 2023"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 204, 'total_tokens': 230, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-Bsp1RHL1g9qNPs4Bw94Mv7MbieKhE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--23a56c4e-7880-4023-85f1-7dcc21298048-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'current captain of the Winnipeg Jets 2023'}, 'id': 'call_B6oBeUQ0crL0oZWbFb6M

Let's look at what happened:

1. Our state object was populated with our request
2. The state object was passed into our entry point (agent node) and the agent node added an `AIMessage` to the state object and passed it along the conditional edge
3. The conditional edge received the state object, found the "tool_calls" `additional_kwarg`, and sent the state object to the action node
4. The action node added the response from the OpenAI function calling endpoint to the state object and passed it along the edge to the agent node
5. The agent node added a response to the state object and passed it along the conditional edge
6. The conditional edge received the state object, could not find the "tool_calls" `additional_kwarg` and passed the state object to END where we see it output in the cell above!

Now let's look at an example that shows a multiple tool usage - all with the same flow!

In [18]:
inputs = {"messages" : [HumanMessage(content="Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!")]}

async for chunk in simple_agent_graph.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        if node == "action":
          print(f"Tool Used: {values['messages'][0].name}")
        print(values["messages"])

        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_5afsA0KZ5iWuBdgu99xFNp1W', 'function': {'arguments': '{"query":"QLoRA"}', 'name': 'arxiv'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 220, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-Bsp1YNGYaSiwzGQajQpC6rWCJDGdf', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--bdbbddd6-6830-41b2-a864-4556f344cc41-0', tool_calls=[{'name': 'arxiv', 'args': {'query': 'QLoRA'}, 'id': 'call_5afsA0KZ5iWuBdgu99xFNp1W', 'type': 'tool_call'}], usage_metadata={'input_tokens': 220, 'output_tokens': 16, 'total_tokens': 236, 'inpu

#### 🏗️ Activity #2:

Please write out the steps the agent took to arrive at the correct answer.

##### ✅ Answer:

Interesting that the activity specifies arriving at the _correct_ answer, because it took many tries to get one! In fact, I ended up seeing in the original output that the model was ran with gpt-4o, not the nano model as was present in the code. Once I used gpt4o, it finally worked with these steps:

1. Entry Point: Our state object was populated with our request: "Search Arxiv for the QLoRA paper, then search each of the authors to find out their latest Tweet using Tavily!" as a `HumanMessage`.

2. Entry Point => Agent Node: The state object was passed into our agent node, and based on the state and tool descriptions, decided to use the Arxiv tool to search for the "QLoRA" paper. When we say "decided to", we mean it created an `AIMessage` (via OpenAI tool calling) that indicated that we should invoke the Arxiv function with a specific query: `"query":"QLoRA"`

3. Agent Node => should continue? => Action: We encounter the conditional edge in which we check if an action should be taken, or if we're done. In the state object, we found the "tool_calls" `additional_kwarg`, meaning we are not done and we continue on to "Action" node.

4. Action Node => Agent Node: In our action node, the state is read and an action is taken. Namely, our tool is finally invoked. Once Arxiv completes it's search, our state is updated with the results and send on to our "Agent" node.

5. Agent Node => should continue? => Action: Our agent looks at our updated state and sees that we have the paper we need. Based on our initial human message, it determines that the next step is to search for tweets for all of the authors listed on the provided QLoRA paper. Again, based on the state and tool descriptions, the agent creates an `AIMessage` with **four** tool calls. We then encounter "should continue" again, and because we have the "tool_calls" `additional_kwarg`, we know we need to take an action. We're not done!

6. Action Node => Agent Node: In our action node, the state is read and an action is taken. The tool calls happen to be four different calls to tavily searching for "Luke Zettlemoyer latest tweet" and the respective searches for the other three authors. Once these searches are completed (in parallel!) we are passed back to the agent with this updated state (the results of our searches).

7. Agent Node => should continue? => END: The agent node evaluates the state, sees that we have everything we need to answer our initial human message, and creates that message as an `AIMessage`. This message is our final message for the user. As always, it adds this to our state. Most importantly, the agent decides **not** to call a tool. We are passed along to our conditional check, we _don't_ have any "tool_calls", and therefore we return `END` and exit our graph. Tada!

# 🤝 Breakout Room #2

## Part 1: LangSmith Evaluator

### Pre-processing for LangSmith

To do a little bit more preprocessing, let's wrap our LangGraph agent in a simple chain.

In [19]:
def convert_inputs(input_object):
  return {"messages" : [HumanMessage(content=input_object["question"])]}

def parse_output(input_state):
  return input_state["messages"][-1].content

agent_chain_with_formatting = convert_inputs | simple_agent_graph | parse_output

In [20]:
agent_chain_with_formatting.invoke({"question" : "What is RAG?"})

"RAG stands for Retrieval-Augmented Generation. It is a technique used in natural language processing (NLP) that combines retrieval-based methods with generative models to improve the quality and accuracy of generated text. Here's how it works:\n\n1. **Retrieval**: The system first retrieves relevant information from a large corpus or database. This step involves searching for documents, passages, or data that are related to the input query or context.\n\n2. **Augmentation**: The retrieved information is then used to augment the input to a generative model. This means that the generative model has access to additional context or facts that can help it produce more accurate and informative responses.\n\n3. **Generation**: Finally, the generative model uses both the original input and the retrieved information to generate a response. This can be in the form of answering questions, completing sentences, or generating text based on a given prompt.\n\nRAG is particularly useful in scenarios

### Task 1: Creating An Evaluation Dataset

Just as we saw last week, we'll want to create a dataset to test our Agent's ability to answer questions.

In order to do this - we'll want to provide some questions and some answers. Let's look at how we can create such a dataset below.

```python
questions = [
    "What optimizer is used in QLoRA?",
    "What data type was created in the QLoRA paper?",
    "What is a Retrieval Augmented Generation system?",
    "Who authored the QLoRA paper?",
    "What is the most popular deep learning framework?",
    "What significant improvements does the LoRA system make?"
]

answers = [
    {"must_mention" : ["paged", "optimizer"]},
    {"must_mention" : ["NF4", "NormalFloat"]},
    {"must_mention" : ["ground", "context"]},
    {"must_mention" : ["Tim", "Dettmers"]},
    {"must_mention" : ["PyTorch", "TensorFlow"]},
    {"must_mention" : ["reduce", "parameters"]},
]
```

#### 🏗️ Activity #3:

Please create a dataset in the above format with at least 5 questions.

##### ✅ Answer:

In [21]:
questions = [
    "What is the ReAct pattern in AI?",
    "Who are the main authors of the ReAct paper?",
    "What does ReAct stand for?",
    "What are the key components of the ReAct framework?",
    "How does ReAct improve upon previous reasoning approaches?",
    "What datasets were used to evaluate ReAct?",
    "What are the main advantages of the ReAct approach?"
]

answers = [
    {"must_mention" : ["reasoning", "acting"]},
    {"must_mention" : ["Yao", "Zhou"]},
    {"must_mention" : ["reasoning", "acting"]},
    {"must_mention" : ["thought", "action", "observation"]},
    {"must_mention" : ["interleaving", "reasoning"]},
    {"must_mention" : ["HotpotQA", "FEVER"]},
    {"must_mention" : ["modular", "interpretable"]},
]

# questions = [
#     "What optimizer is used in QLoRA?",
#     "What data type was created in the QLoRA paper?",
#     "What is a Retrieval Augmented Generation system?",
#     "Who authored the QLoRA paper?",
#     "What is the most popular deep learning framework?",
#     "What significant improvements does the LoRA system make?"
# ]

# answers = [
#     {"must_mention" : ["paged", "optimizer"]},
#     {"must_mention" : ["NF4", "NormalFloat"]},
#     {"must_mention" : ["ground", "context"]},
#     {"must_mention" : ["Tim", "Dettmers"]},
#     {"must_mention" : ["PyTorch", "TensorFlow"]},
#     {"must_mention" : ["reduce", "parameters"]},
# ]

Now we can add our dataset to our LangSmith project using the following code which we saw last Thursday!

In [22]:
from langsmith import Client

client = Client()

dataset_name = f"Retrieval Augmented Generation - Evaluation Dataset - {uuid4().hex[0:8]}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Questions about the ReAct Paper to Evaluate RAG over the same paper."
)

client.create_examples(
    inputs=[{"question" : q} for q in questions],
    outputs=answers,
    dataset_id=dataset.id,
)

{'example_ids': ['ce87560a-bf59-493c-b6db-a95301951e09',
  '69fe34e9-7874-4875-9260-c3fc485a818b',
  '53e8f185-f2fb-4eea-9289-00446f33fcfb',
  'ac77f28e-6ed2-49c7-a1b2-1691b3aa3941',
  '38c10f55-d080-4556-aa7d-67d048445138',
  '9f310b74-3830-4da5-a213-f3d18261f02f',
  'f35a2bfe-848c-4763-a86a-eaf38f316646'],
 'count': 7}

#### ❓ Question #3:

How are the correct answers associated with the questions?

> NOTE: Feel free to indicate if this is problematic or not

##### ✅ Answer:

The questions are associated with the correct answers simply by index.

This is very problematic. It means that if the order of one of the list changes in any way, or we add a question or answer inconsistently, our entire dataset could be incorrect. Depending on two lists being the same length and having the proper order to associate them is quite bad practice.

### Task 2: Adding Evaluators

Now we can add a custom evaluator to see if our responses contain the expected information.

We'll be using a fairly naive exact-match process to determine if our response contains specific strings.

In [23]:
from langsmith.evaluation import EvaluationResult, run_evaluator

@run_evaluator
def must_mention(run, example) -> EvaluationResult:
    prediction = run.outputs.get("output") or ""
    required = example.outputs.get("must_mention") or []
    score = all(phrase in prediction for phrase in required)
    return EvaluationResult(key="must_mention", score=score)

#### ❓ Question #4:

What are some ways you could improve this metric as-is?

> NOTE: Alternatively you can suggest where gaps exist in this method.

##### ✅ Answer:

It's not a very sturdy metric for testing. There is plenty of room for false positives and false negatives. For example:

- A correct response may use slightly different phrases while still being a correct answer, and be marked incorrect because it isn't an exact match. For example, for one question we say that the response must mention "reasoning" and "acting". It's not hard to imagine a correct answer that uses the terms "reason" and "act". The fact that these are the same words in a slightly different form will not be considered by our evaluation.

- An incorrect response may use the terms within "must mention" while being egregiously wrong. As long as the terms are present, our metric will count it as correct.

Implementing LLM as judge to make these evaluations would certainly be an option. To make an immediate improvement, we could at least make our tests case insensitive.

Task 3: Evaluating

All that is left to do is evaluate our agent's response!

In [1]:
experiment_results = client.evaluate(
    agent_chain_with_formatting,
    data=dataset_name,
    evaluators=[must_mention],
    experiment_prefix=f"Search Pipeline - Evaluation - {uuid4().hex[0:4]}",
    metadata={"version": "1.0.0"},
)

View the evaluation results for experiment: 'Search Pipeline - Evaluation - c57d-15ea7584' at:
https://smith.langchain.com/o/17336763-e025-4cab-8ce1-f61b7408e302/datasets/a9e7bc46-66b0-461d-ae9b-c20ac075e015/compare?selectedSessions=6bd4cebc-8bb3-46f9-b990-4221084665f0




0it [00:00, ?it/s]

In [44]:
experiment_results

<ExperimentResults Search Pipeline - Evaluation - 944f-e896c0bd>

## Part 2: LangGraph with Helpfulness:

### Task 3: Adding Helpfulness Check and "Loop" Limits

Now that we've done evaluation - let's see if we can add an extra step where we review the content we've generated to confirm if it fully answers the user's query!

We're going to make a few key adjustments to account for this:

1. We're going to add an artificial limit on how many "loops" the agent can go through - this will help us to avoid the potential situation where we never exit the loop.
2. We'll add to our existing conditional edge to obtain the behaviour we desire.

First, let's define our state again - we can check the length of the state object, so we don't need additional state for this.

In [45]:
class AgentState(TypedDict):
  messages: Annotated[list, add_messages]

Now we can set our graph up! This process will be almost entirely the same - with the inclusion of one additional node/conditional edge!

#### 🏗️ Activity #5:

Please write markdown for the following cells to explain what each is doing.

##### YOUR MARKDOWN HERE

##### ✅ Answer:

At a high level, the next cell is creating a new graph container that will use the same `AgentState` as before, and adds two nodes to the graph.

`graph_with_helpfulness_check = StateGraph(AgentState)`: This lines creates a new graph contain and specifies that the graph will use `AgentState` as its state schema. At this point, our graph has no nodes.

`graph_with_helpfulness_check.add_node("agent", call_model)`: This line adds a node to our graph. The name of the node is "agent" and the function represented by this node is the `call_model` function.

`graph_with_helpfulness_check.add_node("action", tool_node)`: This line, similarly to the above, adds a node to our graph. This node is named "action" and uses the `tool_node` function.

In [46]:
graph_with_helpfulness_check = StateGraph(AgentState)

graph_with_helpfulness_check.add_node("agent", call_model)
graph_with_helpfulness_check.add_node("action", tool_node)

##### YOUR MARKDOWN HERE

##### ✅ Answer:

The following line, `graph_with_helpfulness_check.set_entry_point("agent")` defines where we will "enter" our graph. When we interact with the graph, this is where we will start. We use the name, "agent", which references the node that we created in the previous cell.

In [47]:
graph_with_helpfulness_check.set_entry_point("agent")

##### YOUR MARKDOWN HERE

##### ✅ Answer:

This cell defines our new function we'd like to use to decide if the users original input has been answered. We can compare this to the `should_continue` function. It's going to read the state and return some value to indicate if we should keep going or not, but this time we're using more complex decision making.

Before defining the function, we import `PromptTemplate` which allows us to create a structured prompt with placeholders for dynamic content, and `StrOutputParser` that turns our model's output into plain text.

I'll explain the `tool_call_or_helpful` function in detail in the next text cell since Activity #4 prompts for it there.

In [48]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

def tool_call_or_helpful(state):
  last_message = state["messages"][-1]

  if last_message.tool_calls:
    return "action"

  initial_query = state["messages"][0]
  final_response = state["messages"][-1]

  if len(state["messages"]) > 10:
    return "END"

  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""

  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)

  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")

  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()

  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})

  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"

#### 🏗️ Activity #4:

Please write what is happening in our `tool_call_or_helpful` function!

##### YOUR MARKDOWN HERE

##### ✅ Answer:

When we use `tool_call_or_helpful` function, we take these steps:

1. We take the _last_ message (or most recent) from the end of our list of messages that are stored in the state: `last_message = state["messages"][-1]`

---

2. We check if the last message contains a tool call. In other words, has the agent determined that a tool needs called? **If there is a tool call** we return "action", indicating that we are not done. Keep going.

```python
  if last_message.tool_calls:
    return "action"
```

---

3. Next, (if we haven't returned, of course) we grab the initial query from the state, and again fetch the last message, just with a different variable name:

```python
  initial_query = state["messages"][0]
  final_response = state["messages"][-1]
```

---

4. We make a conditional check to see if we have more than 10 messages. If so, we return "END". This stops us from entering excessively long loops:

```python
  if len(state["messages"]) > 10:
    return "END"
```

---

5. Next, we define a prompt that, in essense, asks "given this initial query and the most recent message, have we done something helpful?

```python
  prompt_template = """\
  Given an initial query and a final response, determine if the final response is extremely helpful or not. Please indicate helpfulness with a 'Y' and unhelpfulness as an 'N'.

  Initial Query:
  {initial_query}

  Final Response:
  {final_response}"""
```

---

6. We turn our string into a prompt template:

```python
  helpfullness_prompt_template = PromptTemplate.from_template(prompt_template)
```

---

7. We prepare our model we'd like to use for this check:

```python
  helpfulness_check_model = ChatOpenAI(model="gpt-4.1-mini")
```

---

8. Next, we define our chain. We first invoke `helpfullness_prompt_template` to fill in our template with data. The return value from this, our prompt, is passed to our `helpfulness_check_model`, and the return value from this is passed to our `StrOutputParser`.

```python
  helpfulness_chain = helpfullness_prompt_template | helpfulness_check_model | StrOutputParser()
```

---

9. Finally, we run our entire pipeline with the correct data. We need to provide an `initial_query` and a `final_response`:

```python
  helpfulness_response = helpfulness_chain.invoke({"initial_query" : initial_query.content, "final_response" : final_response.content})
```

---

10. We read the result. If the result indicates that we have indeed done something helpful, we return "end". If we haven't done something helpful, we return "continue". These values are what we'll use to determine the result of our conditional edge.

```python
  if "Y" in helpfulness_response:
    return "end"
  else:
    return "continue"
```

##### ✅ End to Activity #4

> Note: I'm a bit confused because we started by having markdown explain the cell _below_, but now activity #4 asked about the function in the cell above. I'm going to explain what's happening in the cell below here and continue with that pattern.

In the following cell, we add in our conditional edge. This edge will be used every time we leave the "agent" node. We provide a mapping dictionary that tells us which return values map to which nodes. If `tool_call_or_helpful` returns "continue", we go to the "agent" node. If it returns "action", we continue to the "action" node, and if it returns "end", we return `END` to finish our execution.

In [49]:
graph_with_helpfulness_check.add_conditional_edges(
    "agent",
    tool_call_or_helpful,
    {
        "continue" : "agent",
        "action" : "action",
        "end" : END
    }
)

##### YOUR MARKDOWN HERE

##### ✅ Answer:

In the following cell, we add an edge going from the "action" node to the "agent" node. Every time we leave "action", we go straight to "agent". 

In [50]:
graph_with_helpfulness_check.add_edge("action", "agent")

##### YOUR MARKDOWN HERE

##### ✅ Answer:

In the following cell, we finally compile our graph. This turns it into its executable format, making some validation checks before hand to make sure our graph makes sense.

In [51]:
agent_with_helpfulness_check = graph_with_helpfulness_check.compile()

##### YOUR MARKDOWN HERE

##### ✅ Answer:

In this final cell, we invoke our graph! Our first step is to create our initial state with a Human message, "Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?"

Then, we invoke our graph with `agent_with_helpfulness_check.astream` to run the agent asynchronously with streaming updates. This means we can see some updates of the progress as it happens instead of waiting until the end. We print out the current node and the messages associated with that run.

In [52]:
inputs = {"messages" : [HumanMessage(content="Related to machine learning, what is LoRA? Also, who is Tim Dettmers? Also, what is Attention?")]}

async for chunk in agent_with_helpfulness_check.astream(inputs, stream_mode="updates"):
    for node, values in chunk.items():
        print(f"Receiving update from node: '{node}'")
        print(values["messages"])
        print("\n\n")

Receiving update from node: 'agent'
[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_BYLKRrDsxtN3HHCH0UUzPEWx', 'function': {'arguments': '{"query": "LoRA machine learning"}', 'name': 'arxiv'}, 'type': 'function'}, {'id': 'call_huiSWB7BURa4V48zdyf6p8cK', 'function': {'arguments': '{"query": "Tim Dettmers"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}, {'id': 'call_szPzfwVuXrI0LN2dV8Fi5iaY', 'function': {'arguments': '{"query": "Attention mechanism machine learning"}', 'name': 'arxiv'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 177, 'total_tokens': 248, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_a288987b44', 'id': 'chatcmpl-BsXJ2xos0epgmHVXApj2s0TEXnRAt', '

### Task 4: LangGraph for the "Patterns" of GenAI

Let's ask our system about the 4 patterns of Generative AI:

1. Prompt Engineering
2. RAG
3. Fine-tuning
4. Agents

In [53]:
patterns = ["prompt engineering", "RAG", "fine-tuning", "LLM-based agents"]

In [54]:
for pattern in patterns:
  what_is_string = f"What is {pattern} and when did it break onto the scene??"
  inputs = {"messages" : [HumanMessage(content=what_is_string)]}
  messages = agent_with_helpfulness_check.invoke(inputs)
  print(messages["messages"][-1].content)
  print("\n\n")

**What is Prompt Engineering?**

Prompt engineering is the process of designing and refining prompts—questions or instructions—to elicit specific responses from AI models. It involves crafting the optimal prompts needed to get the best output from generative AI programs, ensuring that the AI understands the context, nuances, and intent behind every query. This practice is crucial for maximizing the effectiveness of large language models (LLMs) and other generative AI tools, which can output conversational text, images, or other media based on the prompts given. Prompt engineering is seen as a bridge ensuring effective human-AI communication, and it requires creativity and problem-solving skills to ask better questions and help AI models learn more effectively.

For more detailed information, you can refer to [this article on Coursera](https://www.coursera.org/articles/what-is-prompt-engineering) or [this guide on DataCamp](https://www.datacamp.com/blog/what-is-prompt-engineering-the-fu